In [ ]:
# Cell 1: Setup and Imports for Section 3.4
import pandas as pd
import numpy as np
import yfinance as yf
import wrds
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sqlalchemy import text
import getpass
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_rows', 100)

print("="*80)
print("SECTION 3.4: ENTITY IDENTIFICATION & DATA LINKAGE")
print("="*80 + "\n")

print("This section covers:")
print("  1. Primary identifiers: Ticker, PERMNO/PERMCO, GVKEY, CUSIP")
print("  2. The Ticker Problem: ticker reuse and reassignment")
print("  3. Linking CRSP & Compustat using CUSIP")
print("  4. Why robust identifiers are essential for research")
print("\n")

In [ ]:
# Cell 2: Connect to WRDS Database
print("="*80)
print("CONNECTING TO WRDS")
print("="*80 + "\n")

print("Please enter your WRDS credentials:")
print("(Your password will be hidden for security)\n")

username = input("Enter WRDS username: ")
password = getpass.getpass("Enter WRDS password: ")

try:
    print("\nConnecting to WRDS...")
    db = wrds.Connection(wrds_username=username, wrds_password=password)
    print("✓ Successfully connected to WRDS!")
    print(f"  Username: {username}")
    print(f"  Connection established\n")
except Exception as e:
    print(f"✗ Connection failed: {e}")
    print("\nPlease check:")
    print("  1. Your WRDS username and password are correct")
    print("  2. Your WRDS account is active")
    print("  3. You have accepted the WRDS data agreement")
    raise

print("="*80)
print("\n")

In [ ]:
# Cell 3: Define and Explain Primary Identifiers
print("="*80)
print("PART A: PRIMARY IDENTIFIERS")
print("="*80 + "\n")

print("Understanding how different databases identify companies:\n")

# Create comparison table
identifiers_info = {
    'Identifier': ['Ticker', 'PERMNO', 'PERMCO', 'GVKEY', 'CUSIP'],
    'Data Source': ['Yahoo Finance', 'CRSP', 'CRSP', 'Compustat', 'All'],
    'Uniquely Identifies': ['Stock symbol', 'Security', 'Company', 'Company', 'Security'],
    'Permanent': ['No (reused)', 'Yes', 'Yes', 'Yes', 'Yes'],
    'Example': ['AAPL', '14593', '7', '001690', '037833100'],
    'Problem': ['Can change/reused', 'CRSP-specific', 'Multiple securities', 'Compustat-specific', 'Complex']
}

df_identifiers = pd.DataFrame(identifiers_info)

print("IDENTIFIER COMPARISON TABLE:")
print("="*80)
display(df_identifiers)
print("\n")

print("KEY DISTINCTIONS:\n")
print("1. TICKER (e.g., 'AAPL')")
print("   - Human-readable stock symbol")
print("   - Used by Yahoo Finance, Google Finance, Bloomberg")
print("   - Problem: Can be reassigned to different companies over time")
print("   - Example: META was 'FB', Twitter was 'TWTR' then delisted")
print("\n")

print("2. PERMNO (CRSP Permanent Security Number)")
print("   - Unique identifier for each security in CRSP")
print("   - Remains constant even if ticker changes")
print("   - Different share classes have different PERMNOs")
print("   - Example: Berkshire Hathaway Class A (19393) vs Class B (66093)")
print("\n")

print("3. PERMCO (CRSP Permanent Company Number)")
print("   - Unique identifier for each company in CRSP")
print("   - All securities of same company share same PERMCO")
print("   - Used to aggregate all share classes")
print("   - Example: All Apple securities have PERMCO = 7")
print("\n")

print("4. GVKEY (Compustat Global Company Key)")
print("   - Unique identifier for each company in Compustat")
print("   - Permanent across name/ticker changes")
print("   - Used for fundamental data linking")
print("   - Example: Apple GVKEY = 001690")
print("\n")

print("5. CUSIP (Committee on Uniform Securities Identification Procedures)")
print("   - 9-character alphanumeric code")
print("   - Standard across all U.S. databases")
print("   - Used as 'master key' to link CRSP and Compustat")
print("   - Example: Apple CUSIP = 037833100")
print("\n")

In [ ]:
# Cell 4: Demonstrate Identifiers with Real Data (FIXED)
print("="*80)
print("DEMONSTRATING IDENTIFIERS: APPLE INC.")
print("="*80 + "\n")

# Get Apple data from Yahoo Finance
print("1. YAHOO FINANCE:")
print("-" * 80)
apple_yf = yf.Ticker('AAPL')
info = apple_yf.info
print(f"   Ticker: {info.get('symbol', 'N/A')}")
print(f"   Company Name: {info.get('longName', 'N/A')}")
print(f"   No permanent identifier available from Yahoo Finance API")
print("\n")

# Get Apple identifiers from CRSP
print("2. CRSP:")
print("-" * 80)
crsp_identifiers_query = """
SELECT DISTINCT permno, permco, ticker, comnam, cusip, ncusip, namedt, nameendt
FROM crsp.dsenames
WHERE ticker = 'AAPL'
  AND nameendt >= '2024-01-01'
ORDER BY namedt DESC
"""

result = db.connection.execute(text(crsp_identifiers_query))
crsp_apple = pd.DataFrame(result.fetchall(), columns=result.keys())

if len(crsp_apple) > 0:
    print("   CRSP Identifiers for Apple:")
    display(crsp_apple)
    
    apple_permno = crsp_apple.iloc[0]['permno']
    apple_permco = crsp_apple.iloc[0]['permco']
    apple_cusip = crsp_apple.iloc[0]['ncusip']
    
    print(f"\n   Key Identifiers:")
    print(f"   - PERMNO: {apple_permno} (unique to this security)")
    print(f"   - PERMCO: {apple_permco} (company-level)")
    print(f"   - CUSIP: {apple_cusip}")
    print("\n")
else:
    print("   No Apple data found in CRSP")
    apple_cusip = '037833100'  # Hardcode for linking example
    apple_permno = 14593
    apple_permco = 7
    print("\n")

# Get Apple identifiers from Compustat (FIXED QUERY)
print("3. COMPUSTAT:")
print("-" * 80)
compustat_identifiers_query = """
SELECT c.gvkey, c.conm, s.tic, c.cusip, c.cik, c.sic
FROM comp.company c
LEFT JOIN comp.security s ON c.gvkey = s.gvkey
WHERE c.cusip = '037833100'
  AND s.tic = 'AAPL'
LIMIT 1
"""

try:
    result = db.connection.execute(text(compustat_identifiers_query))
    comp_apple = pd.DataFrame(result.fetchall(), columns=result.keys())
    
    if len(comp_apple) > 0:
        print("   Compustat Identifiers for Apple:")
        display(comp_apple)
        
        apple_gvkey = comp_apple.iloc[0]['gvkey']
        apple_cusip_comp = comp_apple.iloc[0]['cusip']
        
        print(f"\n   Key Identifiers:")
        print(f"   - GVKEY: {apple_gvkey} (permanent company ID)")
        print(f"   - CUSIP: {apple_cusip_comp} (links to CRSP)")
        print(f"   - CIK: {comp_apple.iloc[0]['cik']} (SEC identifier)")
        print(f"   - SIC: {comp_apple.iloc[0]['sic']} (Industry code)")
        print("\n")
    else:
        print("   No Apple data found in Compustat")
        print("   (This may be normal if company data is in different table)")
        print("\n")
        apple_gvkey = '001690'  # Hardcode for demonstration
        apple_cusip_comp = '037833100'
        
except Exception as e:
    print(f"   Error querying Compustat: {e}")
    print("   Using hardcoded values for demonstration...")
    apple_gvkey = '001690'
    apple_cusip_comp = '037833100'
    comp_apple = pd.DataFrame()
    print("\n")

print("="*80)
print("SUMMARY: APPLE INC. ACROSS DATABASES")
print("="*80)
print(f"{'Database':<15} {'Primary ID':<15} {'Value':<15} {'Secondary IDs'}")
print("-" * 80)
print(f"{'Yahoo Finance':<15} {'Ticker':<15} {'AAPL':<15} {'None (no permanent ID)'}")
if len(crsp_apple) > 0:
    print(f"{'CRSP':<15} {'PERMNO':<15} {str(apple_permno):<15} PERMCO={apple_permco}, CUSIP={apple_cusip}")
if len(comp_apple) > 0 or apple_gvkey:
    print(f"{'Compustat':<15} {'GVKEY':<15} {apple_gvkey:<15} CUSIP={apple_cusip_comp}")
print("-" * 80)
print("\n")

In [ ]:
# Cell 5: The Ticker Problem - Ticker Reuse Example
print("="*80)
print("PART B: THE TICKER PROBLEM - TICKER REUSE")
print("="*80 + "\n")

print("Problem: Ticker symbols can be reassigned to different companies")
print("This creates serious issues for long-term historical analysis\n")

# Example 1: Find a ticker that was reused
ticker_reuse_query = """
SELECT ticker, permno, comnam, namedt, nameendt
FROM crsp.dsenames
WHERE ticker = 'META'
ORDER BY namedt
"""

print("EXAMPLE: Ticker 'META'")
print("-" * 80)
result = db.connection.execute(text(ticker_reuse_query))
meta_history = pd.DataFrame(result.fetchall(), columns=result.keys())

if len(meta_history) > 0:
    print("History of ticker 'META' in CRSP:")
    display(meta_history)
    print("\n")
    print("Observations:")
    print("  - Facebook changed ticker from 'FB' to 'META' in June 2022")
    print("  - Before Facebook, 'META' may have been used by other companies")
    print("  - Using ticker alone would mix data from different companies")
else:
    print("Searching for 'FB' to 'META' transition...")
    fb_query = """
    SELECT ticker, permno, comnam, namedt, nameendt
    FROM crsp.dsenames
    WHERE permno = 15661
    ORDER BY namedt DESC
    """
    result = db.connection.execute(text(fb_query))
    fb_history = pd.DataFrame(result.fetchall(), columns=result.keys())
    display(fb_history)

print("\n")

# Example 2: Find companies that changed tickers
print("EXAMPLE: Companies That Changed Tickers")
print("-" * 80)

ticker_change_query = """
SELECT DISTINCT d1.permno, d1.ticker as old_ticker, d2.ticker as new_ticker,
       d1.comnam, d1.nameendt as change_date
FROM crsp.dsenames d1
JOIN crsp.dsenames d2 
  ON d1.permno = d2.permno 
  AND d1.nameendt = d2.namedt - 1
WHERE d1.ticker != d2.ticker
  AND d1.nameendt >= '2020-01-01'
  AND d1.nameendt <= '2024-12-31'
ORDER BY d1.nameendt DESC
LIMIT 10
"""

print("Recent ticker changes (2020-2024):")
result = db.connection.execute(text(ticker_change_query))
ticker_changes = pd.DataFrame(result.fetchall(), columns=result.keys())

if len(ticker_changes) > 0:
    display(ticker_changes)
    print("\n")
    print("Why This Matters:")
    print("  - A query for historical data using only ticker would miss data")
    print("  - Example: Querying 'META' before 2022 would find wrong company")
    print("  - Solution: Always use PERMNO (CRSP) or GVKEY (Compustat)")
else:
    print("No recent ticker changes found in sample")

print("\n")

In [ ]:
# Cell 6: Demonstrate Impact on Analysis (FIXED)
print("="*80)
print("IMPACT OF TICKER PROBLEM ON ANALYSIS")
print("="*80 + "\n")

print("Scenario: Analyst wants 5-year history of 'META' using only ticker\n")

# Query using ticker only (WRONG approach)
print("APPROACH 1: Query by Ticker Only (WRONG)")
print("-" * 80)

wrong_query = """
SELECT date, prc, vol
FROM crsp.dsf
WHERE permno IN (
    SELECT DISTINCT permno FROM crsp.dsenames WHERE ticker = 'META'
)
AND date >= '2019-01-01'
AND date <= '2024-12-31'
ORDER BY date
LIMIT 10
"""

result = db.connection.execute(text(wrong_query))
wrong_data = pd.DataFrame(result.fetchall(), columns=result.keys())

if len(wrong_data) > 0:
    print("Results (first 10 rows):")
    # Convert decimal to float
    wrong_data['prc'] = wrong_data['prc'].astype(float)
    wrong_data['vol'] = wrong_data['vol'].astype(float)
    display(wrong_data.head(10))
    print("\n⚠️ Problem: This might include data from old company that used 'META'")
    print("   Or it might be Facebook before the ticker change!")
else:
    print("No data returned")

print("\n")

# Query using PERMNO (CORRECT approach) - FIXED
print("APPROACH 2: Query by PERMNO (CORRECT)")
print("-" * 80)

# Get Facebook/Meta's PERMNO (FIXED: added namedt to SELECT)
fb_permno_query = """
SELECT DISTINCT permno, ticker, comnam, namedt
FROM crsp.dsenames
WHERE comnam LIKE '%META PLATFORMS%' OR comnam LIKE '%FACEBOOK%'
ORDER BY namedt DESC
LIMIT 1
"""

result = db.connection.execute(text(fb_permno_query))
fb_permno_data = pd.DataFrame(result.fetchall(), columns=result.keys())

if len(fb_permno_data) > 0:
    fb_permno = fb_permno_data.iloc[0]['permno']
    print(f"Found Facebook/Meta PERMNO: {fb_permno}")
    print(f"Company: {fb_permno_data.iloc[0]['comnam']}")
    print(f"Current/Recent Ticker: {fb_permno_data.iloc[0]['ticker']}\n")
    
    correct_query = f"""
    SELECT date, prc, vol
    FROM crsp.dsf
    WHERE permno = {fb_permno}
    AND date >= '2019-01-01'
    AND date <= '2024-12-31'
    ORDER BY date
    LIMIT 10
    """
    
    result = db.connection.execute(text(correct_query))
    correct_data = pd.DataFrame(result.fetchall(), columns=result.keys())
    
    if len(correct_data) > 0:
        print("Results (first 10 rows):")
        correct_data['prc'] = correct_data['prc'].astype(float)
        correct_data['vol'] = correct_data['vol'].astype(float)
        display(correct_data.head(10))
        print("\n✓ Correct: This is guaranteed to be Facebook/Meta data only")
        print("  regardless of ticker changes (FB → META)")
        print(f"  All data is for PERMNO {fb_permno}, ensuring consistency")
    else:
        print("No price data found for this period")
else:
    print("Facebook/Meta not found in CRSP")
    print("This might be due to database access or naming variations")

print("\n")

# Show ticker history for Facebook/Meta to prove the point
print("VERIFICATION: Ticker History for Facebook/Meta")
print("-" * 80)

if len(fb_permno_data) > 0:
    ticker_history_query = f"""
    SELECT ticker, comnam, namedt, nameendt
    FROM crsp.dsenames
    WHERE permno = {fb_permno}
    ORDER BY namedt DESC
    """
    
    result = db.connection.execute(text(ticker_history_query))
    ticker_history = pd.DataFrame(result.fetchall(), columns=result.keys())
    
    if len(ticker_history) > 0:
        print(f"Complete ticker history for PERMNO {fb_permno}:")
        display(ticker_history)
        print("\nObservation:")
        print("  - Notice the ticker changed from 'FB' to 'META'")
        print("  - But PERMNO remained constant throughout")
        print("  - This is why PERMNO is essential for historical analysis")

print("\n")

print("="*80)
print("KEY TAKEAWAY: THE TICKER PROBLEM")
print("="*80)
print("""
1. NEVER rely on ticker symbols alone for historical analysis
2. Tickers can change due to:
   - Company rebranding (Facebook → Meta, ticker 'FB' → 'META')
   - Mergers and acquisitions
   - Corporate restructuring
   - Ticker reassignment after delisting

3. ALWAYS use permanent identifiers:
   - CRSP: Use PERMNO (security level) or PERMCO (company level)
   - Compustat: Use GVKEY (company level)
   - Cross-database: Use CUSIP to link

4. Yahoo Finance Problem:
   - Only provides ticker-based access
   - No permanent identifier available
   - Historical data may be incorrect if ticker was reused
   - Not suitable for rigorous academic research

5. Real Example (Facebook/Meta):
   - Company changed ticker from 'FB' to 'META' in June 2022
   - Querying by ticker 'FB' after 2022: no recent data
   - Querying by ticker 'META' before 2022: no historical data
   - Querying by PERMNO: complete history regardless of ticker
""")
print("\n")

In [ ]:
# Cell 7: Linking CRSP and Compustat Using CUSIP (FINAL VERSION)
print("="*80)
print("PART C: LINKING CRSP & COMPUSTAT")
print("="*80 + "\n")

print("Challenge: Combine price data (CRSP) with fundamental data (Compustat)")
print("Solution: Use CUSIP as the 'master key' to link databases\n")

print("METHOD 1: Manual Linking Using CUSIP")
print("="*80)

# Select a company to demonstrate
demo_ticker = 'MSFT'
print(f"\nDemonstration: Linking {demo_ticker} data\n")

# Step 1: Get CRSP data with CUSIP
print("Step 1: Get CRSP data with CUSIP")
print("-" * 80)

crsp_link_query = f"""
SELECT d.permno, d.permco, d.ticker, d.comnam, d.ncusip as cusip,
       d.namedt, d.nameendt
FROM crsp.dsenames d
WHERE d.ticker = '{demo_ticker}'
  AND d.nameendt >= '2024-01-01'
"""

result = db.connection.execute(text(crsp_link_query))
crsp_link_data = pd.DataFrame(result.fetchall(), columns=result.keys())

if len(crsp_link_data) > 0:
    print(f"CRSP data for {demo_ticker}:")
    display(crsp_link_data)
    
    cusip_to_link = crsp_link_data.iloc[0]['cusip']
    permno_to_link = crsp_link_data.iloc[0]['permno']
    
    print(f"\nCUSIP to use for linking: {cusip_to_link} (8-digit from CRSP)")
    print("\n")
else:
    print(f"No CRSP data found for {demo_ticker}\n")
    cusip_to_link = None
    permno_to_link = None

# Step 2: Get Compustat data using CUSIP
if cusip_to_link:
    print("Step 2: Get Compustat data using same CUSIP")
    print("-" * 80)
    
    # Join security table (has CUSIP) with company table (has other info)
    comp_link_query = f"""
    SELECT DISTINCT c.gvkey, c.conm, s.cusip, c.sic, s.tic
    FROM comp.company c
    JOIN comp.security s ON c.gvkey = s.gvkey
    WHERE s.cusip LIKE '{cusip_to_link}%'
    LIMIT 1
    """
    
    result = db.connection.execute(text(comp_link_query))
    comp_link_data = pd.DataFrame(result.fetchall(), columns=result.keys())
    
    if len(comp_link_data) > 0:
        print(f"Compustat data for CUSIP {cusip_to_link}:")
        display(comp_link_data)
        
        gvkey_linked = comp_link_data.iloc[0]['gvkey']
        
        print(f"\n✓ Successfully linked!")
        print(f"  CRSP PERMNO {permno_to_link} ← CUSIP {cusip_to_link} → Compustat GVKEY {gvkey_linked}")
        print(f"  Ticker verification: {comp_link_data.iloc[0]['tic']}")
        print(f"  SIC Code: {comp_link_data.iloc[0]['sic']}")
        print("\n")
        
    else:
        print(f"No Compustat data found for CUSIP {cusip_to_link}")
        print("Note: CUSIP format may differ between databases")
        print("  - CRSP uses 8-digit CUSIP (NCUSIP)")
        print("  - Compustat may use 9-digit CUSIP")
        print("  - Manual matching can be unreliable\n")
        gvkey_linked = None
        comp_link_data = pd.DataFrame()

print("\n")

print("METHOD 2: Using CRSP-Compustat Merged Database (CCM) [RECOMMENDED]")
print("="*80)
print("\nWRDS provides a pre-built linking table: CRSP-Compustat Merged (CCM)")
print("This is the standard approach in academic research\n")

# Try to query the CCM link table (requires CCM subscription)
if permno_to_link:
    try:
        ccm_query = f"""
        SELECT gvkey, lpermno as permno, lpermco as permco, 
               linktype, linkprim, linkdt, linkenddt
        FROM crsp.ccmxpf_lnkhist
        WHERE lpermno = {permno_to_link}
        ORDER BY linkdt DESC
        """
        
        result = db.connection.execute(text(ccm_query))
        ccm_link = pd.DataFrame(result.fetchall(), columns=result.keys())
        
        if len(ccm_link) > 0:
            print(f"CCM Link Table for PERMNO {permno_to_link}:")
            display(ccm_link)
            
            print("\nKey Fields Explained:")
            print("  - GVKEY: Compustat company identifier")
            print("  - LPERMNO: CRSP PERMNO (security level)")
            print("  - LPERMCO: CRSP PERMCO (company level)")
            print("  - LINKTYPE: Quality of the link")
            print("      * LC = Link Complete (best quality, manually verified)")
            print("      * LU = Link by CUSIP (unverified, automatic)")
            print("      * LX = Link to non-primary security")
            print("  - LINKPRIM: Primary link indicator")
            print("      * P = Primary (use this one)")
            print("      * C = Secondary")
            print("  - LINKDT/LINKENDDT: Date range when link is valid")
            print("\n")
            
            print("Best Practice for Using CCM Links:")
            print("  1. Filter for LINKTYPE IN ('LC', 'LU')  -- Good quality links")
            print("  2. Filter for LINKPRIM = 'P'  -- Primary links only")
            print("  3. Check date validity: your_date BETWEEN linkdt AND linkenddt")
            print("  4. This ensures you get the correct company match")
            print("\n")
            
            # Compare manual link with CCM link
            if len(comp_link_data) > 0:
                ccm_gvkey = ccm_link.iloc[0]['gvkey']
                print(f"VERIFICATION: Manual vs CCM Linking")
                print("-" * 80)
                print(f"  Manual linking (via CUSIP): GVKEY = {gvkey_linked}")
                print(f"  CCM linking (WRDS table):   GVKEY = {ccm_gvkey}")
                
                if str(gvkey_linked).strip() == str(ccm_gvkey).strip():
                    print(f"  ✓ Perfect Match! Both methods found the same company")
                else:
                    print(f"  ⚠ Different results - CCM is more reliable")
                    print(f"     Manual CUSIP matching can fail due to format differences")
            else:
                print(f"Manual CUSIP linking failed, but CCM link succeeded!")
                print(f"This demonstrates why CCM is the preferred method\n")
        else:
            print(f"No CCM link found for PERMNO {permno_to_link}")
            
    except Exception as e:
        error_msg = str(e)
        if 'permission denied' in error_msg.lower() or 'crsp_a_ccm' in error_msg.lower():
            print("⚠ CCM Database Access Not Available")
            print("-" * 80)
            print("Your WRDS subscription does not include access to the CCM link table.")
            print("\nWhat is CCM?")
            print("  - CRSP/Compustat Merged Database")
            print("  - Pre-validated links between CRSP (stock prices) and Compustat (fundamentals)")
            print("  - Maintained by CRSP and considered the gold standard")
            print("\nAlternatives:")
            print("  1. Use manual CUSIP linking (as shown in Method 1 above)")
            print("  2. Request CCM access from your institution's WRDS administrator")
            print("  3. Use WRDS online query tools which may have CCM built-in")
            print("\nFor this tutorial, we'll continue using manual CUSIP linking.")
        else:
            print(f"Error querying CCM: {error_msg}")
else:
    print("Cannot query CCM - PERMNO not available")

print("\n")

print("="*80)
print("SUMMARY: LINKING METHODS")
print("="*80)
print("""
Method 1: Manual CUSIP Linking (Available)
  ✓ Works with basic CRSP + Compustat access
  ✓ Good for demonstrating concepts
  ✗ CUSIP format differences (8-digit vs 9-digit)
  ✗ Doesn't track historical changes
  ✗ May miss complex corporate actions

Method 2: CCM Link Table (Requires Subscription)
  ✓ Pre-validated by CRSP research team
  ✓ Handles all format differences
  ✓ Tracks historical changes and corporate actions
  ✓ Industry standard for academic research
  ✗ Requires separate CCM subscription

Practical Approach for This Project:
  → Use manual CUSIP linking for our analysis
  → Be aware of potential linking issues
  → For production research, request CCM access
""")
print("\n")

# Save the linked data for later use
if len(comp_link_data) > 0:
    print("Saving linked identifiers for later use...")
    linked_ids = pd.DataFrame({
        'ticker': [demo_ticker],
        'permno': [permno_to_link],
        'gvkey': [gvkey_linked],
        'cusip': [cusip_to_link]
    })
    display(linked_ids)
    print("\nThese identifiers will be used in subsequent cells to merge data.\n")

In [ ]:
# Cell 8: Practical Example - Merging Price and Fundamental Data
print("="*80)
print("PRACTICAL EXAMPLE: MERGING PRICE AND FUNDAMENTAL DATA")
print("="*80 + "\n")

print(f"Goal: Combine {demo_ticker}'s stock prices (CRSP) with financial ratios (Compustat)\n")

# Get recent price data from CRSP
print("Step 1: Get recent price data from CRSP")
print("-" * 80)

price_query = f"""
SELECT date, prc, ret, vol
FROM crsp.dsf
WHERE permno = {permno_to_link}
  AND date >= '2023-01-01'
  AND date <= '2024-12-31'
ORDER BY date DESC
LIMIT 5
"""

result = db.connection.execute(text(price_query))
price_data = pd.DataFrame(result.fetchall(), columns=result.keys())

if len(price_data) > 0:
    # Convert decimal to float
    price_data['prc'] = price_data['prc'].astype(float)
    price_data['ret'] = price_data['ret'].astype(float)
    price_data['vol'] = price_data['vol'].astype(float)
    
    print("Most recent price data:")
    display(price_data)
    print("\n")

# Get fundamental data from Compustat (use gvkey from manual linking)
if len(comp_link_data) > 0:
    gvkey = comp_link_data.iloc[0]['gvkey']
    
    print("Step 2: Get fundamental data from Compustat")
    print("-" * 80)
    
    fundamental_query = f"""
    SELECT datadate, fyear, revt, at, ni, ebitda, ceq
    FROM comp.funda
    WHERE gvkey = '{gvkey}'
      AND datadate >= '2020-01-01'
      AND indfmt = 'INDL'
      AND datafmt = 'STD'
      AND popsrc = 'D'
      AND consol = 'C'
    ORDER BY datadate DESC
    LIMIT 5
    """
    
    result = db.connection.execute(text(fundamental_query))
    fundamental_data = pd.DataFrame(result.fetchall(), columns=result.keys())
    
    if len(fundamental_data) > 0:
        # Convert decimal to float
        for col in ['revt', 'at', 'ni', 'ebitda', 'ceq']:
            if col in fundamental_data.columns:
                fundamental_data[col] = fundamental_data[col].astype(float)
        
        print(f"Recent fundamental data for GVKEY {gvkey}:")
        display(fundamental_data)
        
        print("\nVariable Definitions:")
        print("  - DATADATE: Date of fiscal year end")
        print("  - FYEAR: Fiscal year")
        print("  - REVT: Revenue (Total) - in millions")
        print("  - AT: Assets (Total) - in millions")
        print("  - NI: Net Income - in millions")
        print("  - EBITDA: Earnings Before Interest, Taxes, Depreciation, Amortization")
        print("  - CEQ: Common Equity (Book Value) - in millions")
        print("\n")
        
        # Calculate some basic ratios
        print("Step 3: Calculate Financial Ratios")
        print("-" * 80)
        
        fundamental_data['ROE'] = (fundamental_data['ni'] / fundamental_data['ceq']) * 100
        fundamental_data['ROA'] = (fundamental_data['ni'] / fundamental_data['at']) * 100
        fundamental_data['Asset_Turnover'] = fundamental_data['revt'] / fundamental_data['at']
        
        print("Calculated Profitability Ratios:")
        ratio_df = fundamental_data[['datadate', 'fyear', 'ROE', 'ROA', 'Asset_Turnover']].copy()
        display(ratio_df)
        
        print("\nRatio Interpretations:")
        print("  - ROE (Return on Equity): Net Income / Equity (higher is better)")
        print("  - ROA (Return on Assets): Net Income / Total Assets (higher is better)")
        print("  - Asset Turnover: Revenue / Total Assets (efficiency measure)")
        print("\n")
        
    else:
        print("No fundamental data found\n")
else:
    print("Cannot retrieve fundamental data - linking failed in previous step\n")

print("="*80)
print("LINKED DATA ENABLES ADVANCED ANALYSIS")
print("="*80)
print("""
With linked CRSP-Compustat data, researchers can:

1. Calculate valuation ratios:
   - Price-to-Earnings (P/E) = Market Cap / Net Income
   - Price-to-Book (P/B) = Market Cap / Book Value
   - Enterprise Value / EBITDA

2. Study return predictability:
   - Do firms with low P/E outperform?
   - Does book-to-market ratio predict returns?
   - Fama-French factor research

3. Event studies:
   - Stock price reaction to earnings announcements
   - Impact of dividend changes on returns
   - Merger & acquisition effects

4. Risk analysis:
   - Does leverage predict volatility?
   - Do profitable firms have lower beta?
   - Credit risk and default prediction

5. Corporate finance research:
   - Capital structure decisions
   - Dividend policy
   - Investment and growth

Without proper linking, these analyses would be impossible or unreliable.
The combination of high-frequency price data (CRSP) with annual fundamentals
(Compustat) is the foundation of most empirical corporate finance research.
""")
print("\n")

print("="*80)
print("NEXT STEPS FOR YOUR RESEARCH")
print("="*80)
print("""
Now that you understand the basics:

1. Scale up: Link entire universes of stocks
   - Use CCM for reliable linking at scale
   - Handle date alignment carefully
   - Filter for data quality

2. Time-series analysis:
   - Merge monthly/daily returns with quarterly/annual fundamentals
   - Handle announcement dates vs period dates
   - Deal with restatements and data revisions

3. Cross-sectional analysis:
   - Compare firms within industries (use SIC codes)
   - Control for size, leverage, profitability
   - Handle survivorship bias

4. Portfolio construction:
   - Rank stocks by fundamental metrics
   - Calculate portfolio returns
   - Evaluate performance (Sharpe ratio, alpha, etc.)
""")
print("\n")

In [ ]:
# Cell 9: Visualization - Identifier Relationship Diagram
print("="*80)
print("VISUALIZATION: IDENTIFIER RELATIONSHIPS")
print("="*80 + "\n")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Identifier hierarchy
ax = axes[0, 0]
ax.axis('off')
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)

# Draw boxes for each identifier type
boxes = [
    {'name': 'TICKER\n(AAPL)', 'pos': (5, 9), 'color': '#FFB6C1', 'desc': 'Not Permanent\nCan Change'},
    {'name': 'PERMNO\n(14593)', 'pos': (2, 6), 'color': '#90EE90', 'desc': 'Permanent\nSecurity ID'},
    {'name': 'PERMCO\n(7)', 'pos': (5, 6), 'color': '#90EE90', 'desc': 'Permanent\nCompany ID'},
    {'name': 'GVKEY\n(001690)', 'pos': (8, 6), 'color': '#87CEEB', 'desc': 'Permanent\nCompany ID'},
    {'name': 'CUSIP\n(037833100)', 'pos': (5, 3), 'color': '#FFD700', 'desc': 'Master Key\nLinks All'},
]

for box in boxes:
    rect = plt.Rectangle((box['pos'][0]-0.8, box['pos'][1]-0.5), 1.6, 1, 
                          facecolor=box['color'], edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(box['pos'][0], box['pos'][1], box['name'], 
            ha='center', va='center', fontsize=11, fontweight='bold')
    ax.text(box['pos'][0], box['pos'][1]-0.8, box['desc'], 
            ha='center', va='top', fontsize=8)

# Draw arrows
arrows = [
    ((5, 8.5), (2, 6.5), 'CRSP'),
    ((5, 8.5), (5, 6.5), 'CRSP'),
    ((5, 8.5), (8, 6.5), 'Compustat'),
    ((2, 5.5), (5, 3.5), 'Link'),
    ((5, 5.5), (5, 3.5), 'Link'),
    ((8, 5.5), (5, 3.5), 'Link'),
]

for start, end, label in arrows:
    ax.annotate('', xy=end, xytext=start,
                arrowprops=dict(arrowstyle='->', lw=2, color='darkblue'))
    mid = ((start[0]+end[0])/2, (start[1]+end[1])/2)
    ax.text(mid[0]+0.3, mid[1], label, fontsize=9, style='italic')

ax.set_title('Identifier Hierarchy and Relationships', fontsize=14, fontweight='bold')

# Plot 2: Ticker problem timeline
ax = axes[0, 1]
ax.axis('off')

timeline_text = """
THE TICKER PROBLEM: Example Timeline

2010-2021: Ticker 'FB' → Facebook Inc.
           PERMNO: 15661

2022-Present: Ticker 'META' → Meta Platforms
              PERMNO: 15661 (unchanged!)

Problem:
• Querying 'FB' after 2022 returns no data
• Querying ticker alone misses historical continuity
• Only PERMNO provides complete history

Solution:
• Always use PERMNO for CRSP queries
• Use GVKEY for Compustat queries
• Link via CUSIP when combining databases
"""

ax.text(0.1, 0.5, timeline_text, transform=ax.transAxes, 
        fontsize=10, verticalalignment='center', family='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

ax.set_title('The Ticker Problem: Why Tickers Fail', fontsize=14, fontweight='bold')

# Plot 3: Linking process flowchart
ax = axes[1, 0]
ax.axis('off')
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)

steps = [
    {'text': '1. Start with CRSP\nGet PERMNO & CUSIP', 'y': 8, 'color': '#FFE4B5'},
    {'text': '2. Use CUSIP to Link\nQuery CCM Link Table', 'y': 5.5, 'color': '#E0BBE4'},
    {'text': '3. Get Compustat GVKEY\nAccess Fundamental Data', 'y': 3, 'color': '#B4E7CE'},
]

for step in steps:
    rect = plt.Rectangle((1, step['y']-0.6), 8, 1.2, 
                          facecolor=step['color'], edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(5, step['y'], step['text'], ha='center', va='center', 
            fontsize=11, fontweight='bold')

# Draw connecting arrows
for i in range(len(steps)-1):
    ax.annotate('', xy=(5, steps[i+1]['y']+0.6), xytext=(5, steps[i]['y']-0.6),
                arrowprops=dict(arrowstyle='->', lw=3, color='darkgreen'))

ax.set_title('CRSP-Compustat Linking Process', fontsize=14, fontweight='bold')

# Plot 4: Summary table
ax = axes[1, 1]
ax.axis('off')

summary_table_data = [
    ['Database', 'Primary ID', 'Use Case', 'Linking Method'],
    ['Yahoo Finance', 'Ticker', 'Current prices', 'No linking (limited)'],
    ['CRSP', 'PERMNO', 'Historical prices', 'CUSIP → Compustat'],
    ['Compustat', 'GVKEY', 'Fundamentals', 'CUSIP → CRSP'],
    ['CCM Link', 'PERMNO+GVKEY', 'Integrated', 'Pre-built by WRDS'],
]

table = ax.table(cellText=summary_table_data, cellLoc='left', loc='center',
                 colWidths=[0.2, 0.2, 0.3, 0.3])
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 2.5)

# Style header row
for i in range(4):
    cell = table[(0, i)]
    cell.set_facecolor('#404040')
    cell.set_text_props(weight='bold', color='white')

# Color other rows
colors = ['#E8F4F8', '#FFE4E1', '#E1FFE1', '#FFF9E1']
for i in range(1, len(summary_table_data)):
    for j in range(4):
        table[(i, j)].set_facecolor(colors[i-1])

ax.set_title('Summary: Identifiers and Linking Methods', fontsize=14, fontweight='bold')

plt.suptitle('Section 3.4: Entity Identification & Data Linkage\nVisual Summary', 
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('/home/aditya/FE511_Project/section_3_4_identifiers.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved to: /home/aditya/FE511_Project/section_3_4_identifiers.png")
print("\n")


In [ ]:
# Cell 10: Final Summary and Conclusions
print("="*80)
print("SECTION 3.4: KEY TAKEAWAYS")
print("="*80)
print("""
1. PRIMARY IDENTIFIERS
   ✓ Ticker: Human-readable but NOT permanent
   ✓ PERMNO: CRSP's permanent security identifier  
   ✓ PERMCO: CRSP's permanent company identifier
   ✓ GVKEY: Compustat's permanent company identifier
   ✓ CUSIP: Universal identifier for linking

2. THE TICKER PROBLEM
   ⚠ Tickers can be reused by different companies
   ⚠ Companies change tickers (FB → META)
   ⚠ Yahoo Finance only provides ticker-based access
   ⚠ Historical analysis using tickers alone is unreliable
   
   Solution: Always use permanent identifiers (PERMNO/GVKEY)

3. LINKING CRSP & COMPUSTAT
   Method 1: Manual linking using CUSIP
   Method 2: CRSP-Compustat Merged (CCM) database [RECOMMENDED]
   
   Best Practice:
   - Use CCM link table (crsp.ccmxpf_lnkhist)
   - Filter for LINKTYPE = 'LC' or 'LU'
   - Filter for LINKPRIM = 'P'
   - Check date validity (LINKDT, LINKENDDT)

4. WHY THIS MATTERS FOR YOUR PROJECT
   ✓ Part 1: Shows why Yahoo Finance is inadequate (no PERMNO)
   ✓ Part 2: Enables proper delisting analysis (need PERMNO)
   ✓ Part 3: Allows price adjustment verification across sources
   ✓ Part 4: Makes strategy backtesting accurate (permanent IDs)

5. RESEARCH IMPLICATIONS
   Academic Research: MUST use PERMNO/GVKEY, never ticker alone
   Institutional Use: Requires robust identifier systems
   Retail Trading: Ticker-based is acceptable for current data only
   
   Yahoo Finance Limitation: No permanent identifiers available
   → Not suitable for rigorous historical research
   → CRSP/Compustat are essential for academic work

6. PRACTICAL RECOMMENDATIONS
   For Price Data: Use CRSP with PERMNO
   For Fundamental Data: Use Compustat with GVKEY
   For Combined Analysis: Link via CCM or CUSIP
   For Current Monitoring: Yahoo Finance acceptable
   For Historical Research: Never use Yahoo Finance alone
""")

print("="*80)
print("SECTION 3.4 COMPLETE")
print("="*80)
print("\n")